In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
team_summaries = pd.read_csv("data/raw/Team Summaries.csv")
opp_stats = pd.read_csv("data/raw/Opponent Stats Per Game.csv")
team_stats = pd.read_csv("data/raw/Team Stats Per Game.csv")

team_summaries = team_summaries[(team_summaries['lg'] == 'NBA') & (team_summaries['season'] >= 2003) & (team_summaries['season'] <= 2025)]
opp_stats = opp_stats[(opp_stats['season'] >= 2003) & (opp_stats['season'] <= 2025)]

In [ ]:
team_summaries.head()

Index(['season', 'lg', 'team', 'abbreviation', 'playoffs', 'age', 'w', 'l',
       'pw', 'pl', 'mov', 'sos', 'srs', 'o_rtg', 'd_rtg', 'n_rtg', 'pace',
       'f_tr', 'x3p_ar', 'ts_percent', 'e_fg_percent', 'tov_percent',
       'orb_percent', 'ft_fga', 'opp_e_fg_percent', 'opp_tov_percent',
       'opp_drb_percent', 'opp_ft_fga', 'arena', 'attend', 'attend_g'],
      dtype='object')

In [ ]:
opp_stats = opp_stats[(opp_stats['season'] >= 2003) & (opp_stats['season'] <= 2025)]
opp_stats

,season,lg,team,abbreviation,playoffs,g,mp_per_game,opp_fg_per_game,opp_fga_per_game,opp_fg_percent,...,opp_ft_percent,opp_orb_per_game,opp_drb_per_game,opp_trb_per_game,opp_ast_per_game,opp_stl_per_game,opp_blk_per_game,opp_tov_per_game,opp_pf_per_game,opp_pts_per_game
0,2025,NBA,Atlanta Hawks,ATL,False,60.0,241.7,43.7,90.6,0.482,...,0.803,10.3,34.2,44.5,28.5,9.4,5.1,16.5,19.8,119.5
1,2025,NBA,Boston Celtics,BOS,False,60.0,242.1,40.6,89.8,0.452,...,0.794,10.5,33.8,44.3,24.1,6.9,3.9,13.1,18.3,108.3
2,2025,NBA,Brooklyn Nets,BRK,False,59.0,241.3,40.1,83.7,0.479,...,0.773,10.0,32.7,42.7,26.6,8.3,5.7,15.3,19.2,111.3
3,2025,NBA,Chicago Bulls,CHI,False,60.0,241.3,44.9,95.2,0.471,...,0.790,11.2,35.9,47.1,29.1,8.2,5.2,12.7,16.5,120.3
4,2025,NBA,Charlotte Hornets,CHO,False,58.0,241.3,41.1,88.2,0.466,...,0.783,11.1,34.0,45.1,26.8,8.4,5.5,14.2,18.7,112.8


In [16]:
opponent_totals = pd.read_csv("data/raw/Opponent Totals.csv")
opponent_totals.columns

Index(['season', 'lg', 'team', 'abbreviation', 'playoffs', 'g', 'mp', 'opp_fg',
       'opp_fga', 'opp_fg_percent', 'opp_x3p', 'opp_x3pa', 'opp_x3p_percent',
       'opp_x2p', 'opp_x2pa', 'opp_x2p_percent', 'opp_ft', 'opp_fta',
       'opp_ft_percent', 'opp_orb', 'opp_drb', 'opp_trb', 'opp_ast', 'opp_stl',
       'opp_blk', 'opp_tov', 'opp_pf', 'opp_pts'],
      dtype='object')

In [20]:
team_totals = pd.read_csv("data/raw/Team Totals.csv")
team_totals.columns
# team_totals = team_totals[(team_summaries['lg'] == 'NBA') & (team_summaries['season'] >= 2003) & (team_summaries['season'] <= 2025)]

Index(['season', 'lg', 'team', 'abbreviation', 'playoffs', 'g', 'mp', 'fg',
       'fga', 'fg_percent', 'x3p', 'x3pa', 'x3p_percent', 'x2p', 'x2pa',
       'x2p_percent', 'ft', 'fta', 'ft_percent', 'orb', 'drb', 'trb', 'ast',
       'stl', 'blk', 'tov', 'pf', 'pts'],
      dtype='object')

In [33]:
processed = pd.read_csv("data/processed/Defensive_Stats_Matrix_2003_2025.csv")

In [ ]:
processed['wl'] = processed['w'] / (processed['w'] + processed['l'])
zero_indices = processed.index[processed['w'].isna()]
zero_indices

Index([ 10,  40,  71, 102, 133, 164, 195, 226, 257, 288, 320, 351, 382, 413,
       444, 475, 506, 537],
      dtype='int64')

In [59]:
cleaned = processed.drop(columns=['team', 'w', 'l', 'abbreviation'])
cleaned.index[cleaned['wl'].isna()]

Index([ 10,  40,  71, 102, 133, 164, 195, 226, 257, 288, 320, 351, 382, 413,
       444, 475, 506, 537],
      dtype='int64')

In [ ]:
cleaned.dropna(inplace=True)

Index([], dtype='int64')

In [61]:
cleaned.columns

Index(['season', 'age', 'd_rtg', 'pace', 'opp_e_fg_percent', 'opp_tov_percent',
       'opp_ft_fga', 'drb_per_game', 'stl_per_game', 'blk_per_game',
       'pf_per_game', 'opp_fg_per_game', 'opp_fg_percent', 'opp_x3p_per_game',
       'opp_x3p_percent', 'opp_x2p_per_game', 'opp_x2p_percent',
       'opp_ft_per_game', 'opp_fta_per_game', 'opp_orb_per_game',
       'opp_ast_per_game', 'opp_tov_per_game', 'opp_pts_per_game', 'wl'],
      dtype='object')

In [89]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
A = cleaned.drop(columns=['wl']).to_numpy()
b = pd.DataFrame(cleaned['wl']).to_numpy()

In [90]:
x = np.linalg.lstsq(A, b, rcond=None)[0]    
x

array([[-1.64266252e-03],
       [ 2.56636594e-02],
       [ 1.02537168e-01],
       [ 6.39763086e-03],
       [-1.00282582e+01],
       [-5.28096104e-02],
       [ 1.26464443e+00],
       [ 4.42574458e-02],
       [ 2.31985826e-02],
       [ 1.15731116e-02],
       [-1.49099358e-02],
       [ 5.34716590e-02],
       [-7.48166475e+00],
       [ 6.35514257e-03],
       [-6.46500779e-01],
       [ 1.26276711e-02],
       [ 2.07984996e+00],
       [-6.44930333e-02],
       [ 3.18528061e-02],
       [-9.25525666e-02],
       [-2.70043083e-02],
       [ 1.61985256e-01],
       [-4.42288663e-02]])

In [91]:
error = np.linalg.norm(A @ x - b) / np.linalg.norm(b)
error

0.17794247144326064

In [80]:
precurry = cleaned[cleaned['season'] < 2015]
postcurry = cleaned[cleaned['season'] >= 2015]

In [81]:
A_pre = precurry.drop(columns=['wl']).to_numpy()
b_pre = pd.DataFrame(precurry['wl']).to_numpy()
x_pre = np.linalg.lstsq(A_pre, b_pre, rcond=None)[0]    
error = np.linalg.norm(A_pre @ x_pre - b_pre) / np.linalg.norm(b_pre)
error

0.16128180182889149

In [83]:
A_post = postcurry.drop(columns=['wl']).to_numpy()
b_post = pd.DataFrame(postcurry['wl']).to_numpy()
x_post = np.linalg.lstsq(A_post, b_post, rcond=None)[0]    
error = np.linalg.norm(A_post @ x_post - b_post) / np.linalg.norm(b_post)
error

0.17596513532926444

In [ ]:
m, n = A.shape
U, S, VT = np.linalg.svd(A, full_matrices=True)
A_k = np.zeros((m, n), dtype=float)
errs = []
for u, s, vt in zip(U, S, VT):
    A_k += u*s*vt
    error = np.linalg.norm(A_k @ x - b) / np.linalg.norm(b)
    errs.append(error)


ValueError: operands could not be broadcast together with shapes (688,) (23,) 

In [165]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
features = []
errs = []
for i in range(1, len(cleaned.columns) + 1):
    estimator = LinearRegression()
    selector = RFE(estimator, n_features_to_select=i)
    selector = selector.fit(A, b)
    desired_cols = []
    for i, feat in enumerate(selector.support_):
        if feat:
            desired_cols.append(cleaned.drop(columns=['wl']).columns[i])
    features.append(desired_cols)
    A_reduced = A[:, selector.support_]
    x_reduced = np.linalg.lstsq(A_reduced, b, rcond=None)[0]    
    error = np.linalg.norm(A_reduced @ x_reduced- b) / np.linalg.norm(b)
    errs.append(error)

In [166]:
errs
features 

[['opp_fg_percent'],
 ['opp_e_fg_percent', 'opp_fg_percent'],
 ['opp_e_fg_percent', 'opp_fg_percent', 'opp_x3p_percent'],
 ['opp_e_fg_percent', 'opp_ft_fga', 'opp_fg_percent', 'opp_x3p_percent'],
 ['opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_percent',
  'opp_x3p_percent',
  'opp_x2p_percent'],
 ['opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_per_game',
  'opp_fg_percent',
  'opp_x3p_percent',
  'opp_x2p_percent'],
 ['opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_per_game',
  'opp_fg_percent',
  'opp_x3p_percent',
  'opp_x2p_percent',
  'opp_pts_per_game'],
 ['opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_per_game',
  'opp_fg_percent',
  'opp_x3p_percent',
  'opp_x2p_percent',
  'opp_orb_per_game',
  'opp_pts_per_game'],
 ['opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_per_game',
  'opp_fg_percent',
  'opp_x3p_percent',
  'opp_x2p_percent',
  'opp_orb_per_game',
  'opp_tov_per_game',
  'opp_pts_per_game'],
 ['d_rtg',
  'opp_e_fg_percent',
  'opp_ft_fga',
  'opp_fg_per_game',
  'opp_fg_per